# CSC 2301 · Week 5 — Lists, tuples and dictionaries: organising data by hole

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W5_lists_dicts_drillholes.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

A drilling campaign produces two kinds of records: **collars** (where each hole starts: coordinates,
direction) and **intervals** (what was found down the hole). To answer questions such as "which holes cut a
quartz vein?" or "which hole is closest to DH-01?", you organise the records in the right container.

**What you will practise:** lists (append, slicing, sorting) · tuples and unpacking · sets ·
dictionaries (keys, values, `get`, `items`) · a dictionary of lists · choosing the right structure.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/drillholes_assays.csv', 'drillholes_assays.csv'), ('data/collars.csv', 'collars.csv'), ('tests/w5/q1.py', 'tests/q1.py'), ('tests/w5/q2.py', 'tests/q2.py'), ('tests/w5/q3.py', 'tests/q3.py'), ('tests/w5/q4.py', 'tests/q4.py'), ('tests/w5/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Four containers

| container | written | order | change? | duplicates | typical use |
|---|---|---|---|---|---|
| list | `[1.2, 0.4, 1.2]` | yes | yes | yes | a sequence of measurements |
| tuple | `(512300.0, 745210.5)` | yes | **no** | yes | a fixed record: a coordinate |
| set | `{"granite", "schist"}` | no | yes | **no** | "which different values?" |
| dict | `{"DH-01": 0.42}` | insertion | yes | keys unique | look up a value by its name |

In [ ]:
grades = [0.2, 1.4, 3.1, 0.8]
grades.append(0.05)
print(grades[1:3], sorted(grades, reverse=True)[:2])

collar = (512300.0, 745210.5, 212.0)     # a tuple cannot be modified
e, n, z = collar                         # unpacking
print(e, n, z)

liths = {"granite", "schist", "granite"}
print(liths, "schist" in liths)

mean_grade = {"DH-01": 0.42, "DH-02": 1.05}
mean_grade["DH-03"] = 2.31               # add a key
for hole, g in mean_grade.items():
    print(hole, g)
print(mean_grade.get("DH-09", "no data"))

## 2. The two files

In [ ]:
import csv

with open("drillholes_assays.csv", encoding="utf-8") as f:
    assays = list(csv.DictReader(f))
with open("collars.csv", encoding="utf-8") as f:
    collar_rows = list(csv.DictReader(f))

print(len(assays), "intervals;", len(collar_rows), "collars")
print(collar_rows[0])

## 3. The "group by" pattern

Very often you need **one list per key**. Start with an empty dictionary; for each record, create the
list the first time you meet its key, then append.

In [ ]:
intervals_by_lith = {}
for r in assays:
    lith = r["lithology"]
    if lith not in intervals_by_lith:
        intervals_by_lith[lith] = []
    intervals_by_lith[lith].append(r["hole_id"])

for lith, holes in intervals_by_lith.items():
    print(f"{lith:<12} {len(holes):>3} intervals")

---
## Exercises

### Q1. Collars as tuples

Build `collars`: a dictionary mapping each `hole_id` to a tuple of three floats
`(easting_m, northing_m, elevation_m)`.

In [ ]:
collars = {}
...
collars

In [ ]:
grader.check("q1")

### Q2. A dictionary of lists

Build `grades_by_hole`: each `hole_id` maps to the **list** of its grades in g/t (`au_ppb / 1000`),
in file order.

In [ ]:
grades_by_hole = {}
...
{h: len(v) for h, v in grades_by_hole.items()}

In [ ]:
grader.check("q2")

### Q3. Counting with a dictionary

Build `lith_counts`: lithology → number of intervals of that lithology.

In [ ]:
lith_counts = {}
...
lith_counts

In [ ]:
grader.check("q3")

### Q4. A set to remove repeats

Gold is often in **quartz veins**. Store in `holes_with_vein` the **sorted list** of the holes that cut at
least one `quartz vein` interval, each hole once.

In [ ]:
holes_with_vein = ...
holes_with_vein

In [ ]:
grader.check("q4")

### Q5. The nearest hole

1. Write `distance_m(c1, c2)` that takes two collar tuples and returns the **horizontal** distance between
   them in metres: $\sqrt{(E_1-E_2)^2 + (N_1-N_2)^2}$ (ignore the elevation).
2. Store in `nearest_to_dh01` the hole_id closest to `DH-01` (not DH-01 itself).

In [ ]:
def distance_m(c1, c2):
    ...

nearest_to_dh01 = ...
nearest_to_dh01

In [ ]:
grader.check("q5")

### Going further (not checked)

* Build a **distance table**: a dictionary whose keys are tuples `(hole_a, hole_b)` and whose values are distances.
  Why can a tuple be a dictionary key but a list cannot?
* Which lithology has the highest mean grade? Combine `grades_by_hole`'s idea with `lith_counts`.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()